# Import des modules nécessaires

In [11]:
import pandas as pd
import pathlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import re
import spacy
from spacy.tokenizer import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from pathlib import Path
import sys
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import NMF

In [ ]:
nlp = spacy.load("fr_core_news_lg") # Charger le modèle de langue français de spaCy, qui inclut les vecteurs de mots pré-entraînés pour une meilleure analyse sémantique.

/Users/morganr/Champs_lexicaux_N-A/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dossier_path = Path("/Users/morganr/Champs_lexicaux_N-A/corpus_goncourt") # Chemin vers le dossier contenant les fichiers .txt

donnees = [] # Liste pour stocker les données extraites de chaque fichier

# On boucle sur tous les fichiers .txt du dossier
for fichier in dossier_path.glob("*.txt"):
    with open(fichier, "r", encoding="utf-8") as f:
        contenu = f.read()
        
        # On ajoute les données extraites du fichier à la liste
    donnees.append({
            "nom_fichier": fichier.name, # Nom du fichier
            "texte_brut": contenu, # Contenu brut du fichier
        })
      
         

# Création du tableau de bord (DataFrame)
df = pd.DataFrame(donnees)
print(f"{len(df)} textes chargés avec succès.")

10 textes chargés avec succès.


In [ ]:
def segmenter_texte(text, taille=300): # On segmente le texte en segments de 300 mots
    mots = text.split()
    segments = []
    for i in range(0, len(mots), taille):
        segment = " ".join(mots[i:i+taille])
        if len(segment.split()) >= 100: 
            segments.append(segment)
    return segments

lignes = [] # Liste pour stocker les segments de texte avec leurs titres et IDs

for _, row in df.iterrows():
    titre = row["nom_fichier"] # On utilise le nom du fichier comme titre
    texte = row["texte_brut"] # On segmente le texte en segments de 300 mots
    segments = segmenter_texte(texte) # On ajoute chaque segment à la liste avec son titre et son ID
    
    for j, seg in enumerate(segments):
        
        
        lignes.append({ 
            "titre": titre, # Titre du texte (nom du fichier)
            "segment_id": j, # ID du segment (index du segment dans le texte)
            "texte_segment": seg # Contenu du segment de texte
        })

df_segments = pd.DataFrame(lignes)

df_segments

,titre,segment_id,texte_segment
0,1869_madame_gervaisais_travail.txt,0,"— Quarante scudi ? — Oui, signora. — Cela fait..."
1,1869_madame_gervaisais_travail.txt,1,"suivent… — Oui, signora… Nous, nous nous retir..."
2,1869_madame_gervaisais_travail.txt,2,et trempant la plume dans la boue d’un encrier...
3,1869_madame_gervaisais_travail.txt,3,"sa vie, Jules demandait à Edmond de lui en fai..."
4,1869_madame_gervaisais_travail.txt,4,place des touristes consciencieux lisaient le ...
...,...,...,...
2275,1864_renée_mauperin_travail.txt,219,"à son aise, dans son coin, la tête contre le m..."
2276,1864_renée_mauperin_travail.txt,220,disaient bientôt plus rien ; ils restaient mue...
2277,1864_renée_mauperin_travail.txt,221,trempaient dans la lumière le bord de leurs fe...
2278,1864_renée_mauperin_travail.txt,222,"cinq sous, des joujoux gagnés à des loteries, ..."


# Phase de traitement de textes


In [15]:
stop_word = {
    "bon", "bien", "tout", "toute", "tous", "toutes",
    "grand", "petit", "autre", "même", "seul",
    "faire", "aller", "venir", "voir", "dire", "savoir",
    "vouloir", "pouvoir", "falloir", "prendre", "mettre",
    "tenir", "donner", "parler", "trouver", "rester",
    "sembler", "penser", "avoir", "être",
    "chose", "fois",
    "monsieur", "madame", "mademoiselle", "mme", "mlle",
    "cher", "vrai", "renée", "croire",
}
# On peut ajouter d'autres stop words spécifiques au corpus ou à la langue française selon les besoins.

In [17]:
def nettoyer_texte(texte): #  Fonction pour nettoyer le texte : suppression des stop words, ponctuation, chiffres, espaces, et lemmatisation
    doc = nlp(texte) # Traiter le texte avec spaCy pour obtenir les tokens et leurs propriétés
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower()
        
        if (
            not token.is_stop # Ignorer les stop words (mots vides par défaut dans spaCy)
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces
            and token.pos_ in {"NOUN", "ADJ"} # Ne garder que les noms et adjectifs
            and len(token.lemma_) > 3 # Ne garder que les tokens dont la longueur du lemme est supérieure à 2 caractères
            and lemme not in stop_word
        ):
            tokens.append(lemme) # Ajouter le lemme du token en minuscules à la liste des tokens nettoyés
    
    return " ".join(tokens) # Retourner le texte nettoyé en joignant les tokens avec des espaces

df_segments["texte_nettoye"] = df_segments["texte_segment"].apply(nettoyer_texte) # Application de la fonction de nettoyage à chaque segment de texte

df_segments[["texte_segment", "texte_nettoye"]].head() # Affichage des 5 premiers segments avant et après nettoyage



,texte_segment,texte_nettoye
0,"— Quarante scudi ? — Oui, signora. — Cela fait...",scudi monnaie cent franc cent franc romaine ap...
1,"suivent… — Oui, signora… Nous, nous nous retir...",chambre fond besoin mère romaine vieux femme s...
2,et trempant la plume dans la boue d’un encrier...,plume boue encrier reçu étranger carte gervais...
3,"sa vie, Jules demandait à Edmond de lui en fai...",lecture haut mémoire idée manie matin soir fig...
4,place des touristes consciencieux lisaient le ...,place touriste consciencieux guide assiette so...


In [18]:
df_segments["nb_tokens_nettoyes"] = df_segments["texte_nettoye"].apply(lambda x: len(x.split()))
df_segments["nb_tokens_nettoyes"].describe()

count    2280.000000
mean       74.543860
std        17.306285
min        23.000000
25%        62.000000
50%        76.000000
75%        87.000000
max       121.000000
Name: nb_tokens_nettoyes, dtype: float64

In [19]:
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.8
    )

X = vectorizer.fit_transform(df_segments["texte_nettoye"])

X.shape

(2280, 4387)

In [ ]:
def afficher_topics(model, feature_names, n_top_words=10):

    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1} : {' | '.join(top_words)}")
        
        
feature_names = vectorizer.get_feature_names_out()

for k in [8, 10, 12]:

    print(f"\n --NMF avec {k} topics--")

    nmf = NMF(n_components=k,
              init="nndsvda",
              random_state=42,
              max_iter=500
              )

    W = nmf.fit_transform(X)
    afficher_topics(nmf, feature_names, 10)

    print("Erreur :", nmf.reconstruction_err_)


 --NMF avec 8 topics--
Topic 1 : fille | jeune | père | maréchal | mariage | monde | année | salon | femme | amie
Topic 2 : ciel | arbre | soleil | ombre | lumière | jour | blanc | bois | terre | bleu
Topic 3 : sœur | malade | salle | interne | hôpital | médecin | jour | garde | voix | mort
Topic 4 : franc | argent | homme | journal | idée | lettre | atelier | cent | monde | dîner
Topic 5 : femme | amour | homme | pensée | cœur | esprit | parole | maîtresse | nature | passion
Topic 6 : main | tête | femme | oeil | bras | cheveu | corps | pied | noir | blanc
Topic 7 : frère | cirque | exercice | pied | tour | saut | directeur | aîné | jour | clown
Topic 8 : enfant | mère | fils | pauvre | père | jour | cœur | oeil | famille | maman
Erreur : 46.06103429626386

 --NMF avec 10 topics--
Topic 1 : enfant | mère | fils | pauvre | père | cœur | oeil | famille | jour | main
Topic 2 : ciel | arbre | soleil | ombre | bois | blanc | lumière | bleu | terre | noir
Topic 3 : sœur | malade | salle | 

In [23]:
nmf = NMF(n_components=12, 
          random_state=42,
          init="nndsvda",
          solver="cd",
          max_iter=600,
          
          )
W = nmf.fit_transform(X)
H = nmf.components_


feature_names = vectorizer.get_feature_names_out()

def afficher_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1} : {' | '.join(top_words)}")

afficher_topics(nmf, feature_names, 10)

Topic 1 : pensée | cœur | mort | parole | souffrance | amour | jour | idée | maladie | corps
Topic 2 : ciel | arbre | soleil | ombre | bois | blanc | lumière | bleu | terre | noir
Topic 3 : sœur | malade | salle | interne | hôpital | garde | médecin | voix | service | main
Topic 4 : enfant | mère | fils | pauvre | père | cœur | famille | jour | oeil | maman
Topic 5 : heure | jour | chambre | dîner | temps | matin | maison | nuit | porte | soir
Topic 6 : main | tête | oeil | bras | cheveu | noir | pied | blanc | corps | regard
Topic 7 : frère | cirque | exercice | pied | tour | saut | aîné | tonneau | clown | directeur
Topic 8 : fille | jeune | père | maréchal | mariage | monde | année | salon | amie | petite
Topic 9 : homme | journal | idée | talent | tableau | artiste | œuvre | monde | peintre | atelier
Topic 10 : femme | homme | amour | maîtresse | mari | amant | monde | ménage | créature | espèce
Topic 11 : théâtre | acteur | scène | pièce | rôle | tragédien | loge | directeur | fra

In [21]:
nmf = NMF(
    n_components=12,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

W = nmf.fit_transform(X)

df_segments["topic_dominant"] = W.argmax(axis=1)
df_segments["poids_topic"] = W.max(axis=1)

In [22]:
for topic in sorted(df_segments["topic_dominant"].unique()):
    print(f"\n===== TOPIC {topic} =====")
    subset = (
        df_segments[df_segments["topic_dominant"] == topic]
        .sort_values("poids_topic", ascending=False)
        .head(5)
    )
    for _, row in subset.iterrows():
        print(f"\nTitre : {row['titre']} | Segment : {row['segment_id']} | Poids : {row['poids_topic']:.4f}")
        print(row["texte_segment"][:500])


===== TOPIC 0 =====

Titre : 1861_sœur_philomène_travail.txt | Segment : 32 | Poids : 0.1273
et s’amaigrissaient. Un malaise général, des souffrances qui se déplaçaient tous les jours lui donnaient continuellement un sentiment douloureux de toutes les parties de son corps, la conscience et la fatigue du jeu laborieux de ses organes, du travail de la vie dans son être. Elle était abattue en se levant, faible d’une faiblesse qu’elle ne pouvait surmonter. Quand elle montait des escaliers ou qu’elle courait, elle avait des battements de cœur : lui fallait s’asseoir. Le moindre travail lui d

Titre : 1882_la_faustin_travail.txt | Segment : 200 | Poids : 0.1234
elle mangerait, elle n'aurait plus son visage en face d'elle ; et, quand elle dormirait, elle n'aurait plus son sommeil lié au sien ; et elle n'aurait plus sa parole pour dire la même pensée que celle venue au même moment dans sa tête ; et elle n'aurait plus ses yeux pour voir, pour voir à deux... Non, plus rien désormais dans sa vie